#  Supercomputing Internet LLM Fine-Tuning LoRA Example

This notebook is an English, GitHub-friendly translation of the original document. The Python code cells are kept unchanged as requested.


## Overview

This notebook fine-tunes the model `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B` (1.5B parameters) with **LoRA** on a sentiment-analysis dataset.

Key points of this tutorial:

- The base model is downloaded **through the Hugging Face mirror** (`hf-mirror.com`) and saved to `D:\HuggingfaceDownload\DeepSeek-R1-Distill-Qwen-1.5B`.
- Fine-tuning uses 4-bit quantization (NF4) + LoRA so the whole run fits in ~6 GB of GPU memory.
- Everything runs **locally** with the standard Hugging Face `Trainer` workflow (no cloud platform required).
- The dataset `twitter-airline-sentimentSentiment_Analysis.csv` is already downloaded next to this notebook.

This is public-interest code, generated with the help of DeepSeek and tested in an example environment.

## 1. Market context for large language models

Outside of API usage from major frontier vendors, most civilian LLM research and applications are already clearly differentiated at the 500B scale and below (excluding TAALAS-related techniques).

1. **Hundreds-of-billions-scale models (100B+ to 500B)** represented by OpenAI GPT-OSS-120B. This track currently emphasizes low bit precision, such as 4-bit native training precision, strong information encoding, and improved information efficiency without sacrificing quality as the baseline. For example, GPT-OSS-120B can be used as a benchmark and compared directly with GPT-4-class models. Without strong model optimization or theoretical support, domestic models in China may not challenge trillion-parameter models; even if they do, they may still find it difficult to compete with GPT-OSS-120B. This reflects the current mathematical and practical limitations of large-model development in China, not a denial of 10T- or 100T-parameter models. However, hundreds-of-billions-scale models usually require multi-GPU operation, which creates a technical barrier.

2. **Tens-of-billions-scale models** represented by OpenAI GPT-OSS-20B. This group includes the classic Microsoft Phi-4 series at 14B and 7B, as well as distilled models from the Qwen and DeepSeek families, which are mainstream offerings from first- and second-tier vendors. With industrial 4-bit support, these models can run directly on consumer-grade computers (30–50 GB). Their token generation speed is comparable to a single user's information-processing speed. In particular, their fine-tuning compute requirements are relatively small, making them suitable for small and medium-sized enterprises to perform secondary development directly at the model layer. In some cases, the cost can be as low as a few thousand RMB. After efficiency improvements or sub-4-bit optimization, they may become mainstream for edge computing.

3. **Billion-scale models** represented by DeepSeek. A classic example is the DeepSeek 1.5B distilled model, which usually occupies 3 GB of memory or less. After further optimization, these models can be deployed directly on mobile phones or small computers. Their technical parameters are comparable to tens-of-billions-scale models, making them suitable for teaching purposes with extremely low training cost, typically within tens of RMB. The techniques learned here can be quickly transferred to tens-of-billions-scale model workflows, making this an effective way to experiment and iterate.


## 2. Local setup steps

The following steps set up a **local** fine-tuning environment (Windows + Python 3.12, e.g. the `agentic_ai` kernel in VS Code).

1. Install the required libraries (already installed in this environment):

   ```
   pip install torch transformers accelerate peft bitsandbytes datasets trl scikit-learn pandas
   ```

   If you are in China, you can speed up pip with the Aliyun mirror:

   ```
   pip config set global.index-url https://mirrors.aliyun.com/pypi/simple/
   ```

2. **Download the base model through the Hugging Face mirror** (`hf-mirror.com`) into `D:\HuggingfaceDownload\...`:

   The first code cell (Section 3) sets `HF_ENDPOINT=https://hf-mirror.com` and `HF_HOME=D:\HuggingfaceDownload\.cache\huggingface` **before** importing `transformers`, so every model download goes through the mirror and the cache lands on the `D:` drive. The unpacked model is then saved to `D:\HuggingfaceDownload\DeepSeek-R1-Distill-Qwen-1.5B`.

3. **Dataset**: the CSV file `twitter-airline-sentimentSentiment_Analysis.csv` has already been downloaded to

   ```
   C:\Deepin\Programming\20260803 AgenticAILLMVisionModel2026Tutorials\tutorials\01-llm-transformer-training\twitter-airline-sentimentSentiment_Analysis.csv
   ```

   which is the same folder as this notebook, so the code reads it with a relative path (with an absolute-path fallback).

4. Run the notebook cells in order:

   - Sections 3–5: download and test the base model.
   - Sections 6–7: load the dataset and prepare the fine-tuning data.
   - Sections 8–13: 4-bit quantization, LoRA setup, and training.
   - Sections 14–15: test the fine-tuned model.

## 3. Download the base model

Download the model `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B` **through the Hugging Face mirror** (`HF_ENDPOINT=https://hf-mirror.com`) and save it locally to:

`D:\HuggingfaceDownload\DeepSeek-R1-Distill-Qwen-1.5B`

The download cache also lives on the `D:` drive (`D:\HuggingfaceDownload\.cache\huggingface`).

In [1]:
# =============================================================================
# SECTION 3: Download the base model from Hugging Face mirror
# =============================================================================
# This cell downloads DeepSeek-R1-Distill-Qwen-1.5B (~3 GB) via the
# hf-mirror.com mirror (accessible in China) and saves it locally so
# subsequent runs can load it without any network access.

import os

# ---- Use the Hugging Face mirror (hf-mirror.com) for model downloads ----
# IMPORTANT: must be set BEFORE importing transformers / huggingface_hub
# HF_ENDPOINT: redirects all HF downloads to the China-accessible mirror
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
# HF_HOME: stores cached model blobs on the D: drive to save C: drive space
os.environ["HF_HOME"] = r"D:\HuggingfaceDownload\.cache\huggingface"



# Import core Hugging Face classes — AutoModelForCausalLM loads any
# GPT-style (decoder-only) language model; AutoTokenizer loads its tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Define local save directory for the 1.5B model
# After download, this folder will contain config.json, model.safetensors,
# tokenizer.json, and other necessary files
local_model_dir = r"D:\HuggingfaceDownload\DeepSeek-R1-Distill-Qwen-1.5B"

# ✅ Correct Model Name (1.5 Billion parameters)
# DeepSeek-R1-Distill-Qwen-1.5B is a distilled, smaller version of the
# DeepSeek-R1 reasoning model, based on the Qwen2 architecture
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

print(f"Downloading via HF mirror: {os.environ['HF_ENDPOINT']}")
print(f"Model: {model_name} (Official size: 1.54B parameters)")

# Check whether the model was already downloaded — skip if it exists locally
if os.path.isdir(local_model_dir) and os.listdir(local_model_dir):
    print(f"Local model already exists at {local_model_dir}, skipping download...")
else:
    # Download the tokenizer first — it's small (~a few MB)
    # trust_remote_code=True is needed because DeepSeek models use custom code
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    # Download the full model: bfloat16 halves VRAM usage vs float32
    # device_map="auto" lets Hugging Face pick GPU first, then CPU
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,   # bfloat16 is safe and memory efficient
        device_map="auto"             # automatic device placement
    )

    # Persist both tokenizer and model to disk for future offline use
    tokenizer.save_pretrained(local_model_dir)
    model.save_pretrained(local_model_dir)
    print(f"Model saved to {local_model_dir}")

c:\Users\ycasi\anaconda3\envs\agentic_ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model: deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B (Official size: 1.54B parameters)
Local model already exists at D:\HuggingfaceDownload\DeepSeek-R1-Distill-Qwen-1.5B, skipping download...


## 4. Hugging Face mirror notes

Because the model is downloaded through the mirror (`hf-mirror.com`), a few points are worth knowing:

### 4.1 Environment variables

| Variable | Value in this notebook | Effect |
| --- | --- | --- |
| `HF_ENDPOINT` | `https://hf-mirror.com` | All downloads use the mirror instead of `huggingface.co` |
| `HF_HOME` | `D:\HuggingfaceDownload\.cache\huggingface` | Cache and local config live on the `D:` drive |

Both are set with `os.environ[...]` in the first code cell, **before** `transformers` is imported.

### 4.2 Where files are stored

- The mirror cache (blobs): `D:\HuggingfaceDownload\.cache\huggingface\hub`
- The unpacked model used by the rest of this notebook: `D:\HuggingfaceDownload\DeepSeek-R1-Distill-Qwen-1.5B`

### 4.3 Work fully offline afterwards

Once the model is saved locally, the notebook loads it from the local directory, so no network is needed. To make `huggingface_hub` fail loudly instead of trying the network:

```powershell
# in the terminal (PowerShell)
$env:HF_HUB_OFFLINE = "1"
```

### 4.4 Find large files in the download directory

```powershell
Get-ChildItem -Recurse D:\HuggingfaceDownload | Sort-Object Length -Descending | Select-Object -First 20 FullName, Length
```

This shows the 20 largest files and folders under `D:\HuggingfaceDownload`.

## 5. Test the downloaded model

Before fine-tuning, we verify that the base model loads correctly from the local directory and can generate coherent text. This is a **sanity check** — we ask the model a general-knowledge question ("Explain quantum computing in simple terms") and inspect the output. If this works, we know the model files are intact and the environment is properly configured.

The cell below:
1. Loads the tokenizer and model from the local `D:\HuggingfaceDownload\...` directory (no network needed)
2. Tokenizes a prompt string into token IDs
3. Runs autoregressive generation with `model.generate()`
4. Decodes the output tokens back into readable text

In [2]:
# =============================================================================
# SECTION 5: Sanity-check the downloaded model with a simple text generation
# =============================================================================
# This cell loads the locally saved model and runs a single forward generation
# to confirm everything works before we proceed to fine-tuning.

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load the model from the local directory (no network needed)
local_model_dir = r"D:\HuggingfaceDownload\DeepSeek-R1-Distill-Qwen-1.5B"

# Re-load tokenizer and model — this time from local disk, not from HF hub
tokenizer = AutoTokenizer.from_pretrained(local_model_dir, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    local_model_dir,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Example prompt — a general-knowledge question to test the base model
input_text = "Explain quantum computing in simple terms"
# Tokenize: convert string → tensor of token IDs, move to same device as model
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

# Generate text using the base (not yet fine-tuned) model
# max_new_tokens=200: generate up to 200 new tokens after the prompt
# temperature=0.7: moderate randomness (0=deterministic, 1=creative)
# do_sample=True: use sampling instead of greedy decoding
outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id  # use EOS token for padding
)

# Decode the generated token IDs back into human-readable text
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 339/339 [00:01<00:00, 300.81it/s]


Explain quantum computing in simple terms.

Quantum computing is a type of computer that uses quantum-mechanical phenomena, such as superposition and entanglement, to perform operations on data. Unlike classical computers, which use bits to represent information, quantum computers use quantum bits, or qubits, which can exist in multiple states simultaneously. This allows quantum computers to perform certain calculations much faster than classical computers.


## 6. Load the dataset

We use the **Twitter Airline Sentiment** dataset — ~14,000 tweets about US airlines, each labeled as *positive*, *negative*, or *neutral*.

### Train / Validation / Test Split

Following the industry standard, we split the first **7,000 rows** into:

| Split | Rows | Purpose |
|---|---|---|
| **Train** | 5,000 | Used by the optimizer to update model weights |
| **Val** | 1,000 | Used each epoch to track `eval_loss`; best checkpoint is picked by lowest val loss |
| **Test** | 1,000 | **Held out** — never seen during training; used for final evaluation in Sections 14–15 |

The dataset columns are:
- `tweet_id` — unique identifier
- `sentiment` — the label we want the model to predict (positive/negative/neutral)
- `author` — username of the tweet author
- `content` — the tweet text (our model input)

The cell below reads the CSV with pandas, splits the data, and prints a preview.

In [1]:
# =============================================================================
# SECTION 6: Load the Twitter Airline Sentiment dataset
# =============================================================================
# This dataset contains ~14,000 tweets about US airlines, each labeled with
# one of three sentiments: positive, negative, or neutral.
#
# We split the first 7,000 rows into:
#   - Train:   5,000 samples (for fine-tuning)
#   - Eval:    1,000 samples (for validation during training)
#   - Test:    1,000 samples (held-out, for final evaluation in Sections 14-15)
#
# This split follows the industry standard: train/validation/test.

import os
import pandas as pd

# The CSV has already been downloaded next to this notebook.
# Try the relative path first, then fall back to the absolute path.
csv_path = "twitter-airline-sentimentSentiment_Analysis.csv"
if not os.path.exists(csv_path):
    csv_path = r"C:\Deepin\Programming\20260803 AgenticAILLMVisionModel2026Tutorials\tutorials\01-llm-transformer-training\twitter-airline-sentimentSentiment_Analysis.csv"

# Read the CSV into a pandas DataFrame
df = pd.read_csv(csv_path)

# Split first 7,000 rows: 5,000 train | 1,000 eval | 1,000 test
train_df = df.iloc[:5000]          # rows 0-4999
eval_df  = df.iloc[5000:6000]      # rows 5000-5999  (validation during training)
test_df  = df.iloc[6000:7000]      # rows 6000-6999  (held-out, never seen)

print(f"Train: {len(train_df)} rows | Eval: {len(eval_df)} rows | Test: {len(test_df)} rows")
# Preview the first few rows of the training set
print(train_df[['tweet_id', 'sentiment', 'author', 'content']].head())

Train: 5000 rows | Eval: 1000 rows | Test: 1000 rows
     tweet_id   sentiment       author  \
0  1956967341       empty   xoshayzers   
1  1956967666     sadness    wannamama   
2  1956967696     sadness    coolfunky   
3  1956967789  enthusiasm  czareaquino   
4  1956968416     neutral    xkilljoyx   

                                             content  
0  @tiffanylue i know  i was listenin to bad habi...  
1  Layin n bed with a headache  ughhhh...waitin o...  
2                Funeral ceremony...gloomy friday...  
3               wants to hang out with friends SOON!  
4  @dannycastillo We want to trade with someone w...  


The CSV file already exists in the same folder as this notebook:

`C:\Deepin\Programming\20260803 AgenticAILLMVisionModel2026Tutorials\tutorials\01-llm-transformer-training\twitter-airline-sentimentSentiment_Analysis.csv`

## 7. Prepare the fine-tuning data — Prompt-Format Alignment

This is where the **alignment** happens. The key principle: the model must see the **exact same prompt structure** at training time and inference time. Any mismatch will degrade performance.

### The Instruction Template (Training Format)

We wrap every tweet+label pair in a fixed instruction template:

```
Instruction: Analyze the sentiment of the following tweet:
Input: <tweet text>
Output: <sentiment label>
```

This three-part structure teaches the model a specific pattern:
1. **Instruction** — tells the model *what task* to perform
2. **Input** — provides the *data* to analyze (the tweet content)
3. **Output** — shows the *expected answer* (the sentiment label)

### Train / Val / Test Separation

We build **two** Hugging Face `Dataset` objects (`train_dataset`, `val_dataset`) from `train_df` and `eval_df`. The `test_df` stays as a raw DataFrame — it is never seen by the `Trainer`, ensuring unbiased evaluation.

### Loss Masking (Section 9)

During tokenization, we apply **loss masking**: only the tokens after `Output:` contribute to the loss. This prevents the model from learning template boilerplate, which would cause hallucinated text after the sentiment label. See Section 9.1 for the full mathematical explanation.

In [2]:
# =============================================================================
# SECTION 7: Prompt-format alignment — build train/val/test datasets
# =============================================================================
# CRITICAL CONCEPT: The prompt format used here MUST match the format used
# at inference time (in Section 14). This is the alignment constraint:
# the model learns to map "Instruction:\nInput:\nOutput:" → sentiment label.
# If the inference prompt differs even slightly (different wording, missing
# newlines, extra spaces), the model may produce garbage.
#
# We build THREE separate datasets from the splits defined in Section 6:
#   - train_dataset: used by the Trainer for weight updates
#   - val_dataset:   used by the Trainer to compute eval_loss each epoch
#   - test_df:       kept as raw DataFrame for inference in Sections 14-15
#
# The test set is NEVER tokenized or seen by the Trainer — it's held out.
#
# NOTE: Loss masking (Section 9) ensures the model only learns the label,
# not the template boilerplate. See Section 9.1 for the full math explanation.

# The instruction tells the model what task to perform
instruction = "Analyze the sentiment of the following tweet:"

def build_dataset(df, desc):
    """Build a Hugging Face Dataset from a DataFrame using the fixed template."""
    texts = []
    for idx, row in df.iterrows():
        text = f"Instruction: {instruction}\nInput: {row['content']}\nOutput: {row['sentiment']}"
        texts.append(text)
    from datasets import Dataset
    ds = Dataset.from_dict({"text": texts})
    print(f"{desc}: {len(ds)} examples")
    return ds

train_dataset = build_dataset(train_df, "Train")
val_dataset   = build_dataset(eval_df,  "Val")
# test_df is kept as a raw DataFrame — NOT converted to a Dataset
# (it will be used directly for inference in Sections 14-15)

# Print the first training example to verify the format
print("\nFirst training example:")
print(train_dataset[0]['text'])

c:\Users\ycasi\anaconda3\envs\agentic_ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train: 5000 examples
Val: 1000 examples

First training example:
Instruction: Analyze the sentiment of the following tweet:
Input: @tiffanylue i know  i was listenin to bad habit earlier and i started freakin at his part =[
Output: empty


## 8. 4-bit quantization and LoRA model setup

This is the core of the memory-efficient fine-tuning strategy. Three techniques work together:

### 8.1 4-bit NF4 Quantization (BitsAndBytes)
The base model's 1.5 billion float16 parameters (~3 GB) are compressed to 4-bit integers (~0.75 GB). During computation, weights are temporarily dequantized back to bfloat16. The NF4 (NormalFloat4) data type is optimized for normally-distributed weights and outperforms plain int4.

### 8.2 LoRA (Low-Rank Adaptation)
Instead of updating all 1.5B parameters, LoRA inserts small trainable matrices ($A$ and $B$) into the attention layers. Only these matrices are updated during training — the original weights stay **frozen**.

The forward pass for a LoRA-adapted linear layer $W_0 \in \mathbb{R}^{d \times d}$:

$$\mathbf{y} = W_0 \mathbf{x} + \frac{\alpha}{r} B A \mathbf{x}$$

where $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times d}$, and $r \ll d$. The scaling factor $\frac{\alpha}{r}$ controls the magnitude of the LoRA update:

$$\text{Effective LR}_{\text{LoRA}} = \frac{\alpha}{r} \cdot \eta = \frac{32}{8} \cdot (2 \times 10^{-4}) = 8 \times 10^{-4}$$

Key parameters:
- **r (rank)** = 8: Trainable parameters per layer: $d \times r + r \times d = 2dr$ instead of $d^2$
- **lora_alpha** = 32: Higher alpha → larger LoRA update magnitude
- **target_modules**: `q_proj` and `v_proj` (query and value projections in self-attention)

### 8.3 Gradient Checkpointing
Enabled after LoRA wrapping. Trades ~20% slower training for ~30% VRAM savings by recomputing intermediate activations during the backward pass instead of storing them in memory. A standard technique for fitting larger models on consumer GPUs.

The result: only ~0.1% of parameters are trainable, making fine-tuning possible on a single consumer GPU with ~6 GB VRAM.

In [3]:
# =============================================================================
# SECTION 8: 4-bit quantization + LoRA setup
# =============================================================================
# Two key techniques make fine-tuning a 1.5B model possible on ~6 GB VRAM:
# 1. 4-bit NF4 quantization — compresses the base model weights from 16-bit
#    to 4-bit, reducing memory ~4×.
# 2. LoRA (Low-Rank Adaptation) — instead of updating all 1.5B parameters,
#    we only train small "adapter" matrices (~0.1% of the total parameters).
#    The base weights stay frozen.

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig          # ← configures 4-bit / 8-bit quantization
)
from peft import LoraConfig, get_peft_model   # ← PEFT = Parameter-Efficient Fine-Tuning

# Check if a CUDA-capable GPU is available
use_cuda = torch.cuda.is_available()
print(f"CUDA available: {use_cuda}")

# ---------------------------------------------------------------------------
# 4-bit quantization config — only works on GPU (CUDA)
# ---------------------------------------------------------------------------
# - load_in_4bit=True: weights are stored as 4-bit integers
# - bnb_4bit_quant_type="nf4": "NormalFloat4" — better than plain int4
# - bnb_4bit_compute_dtype=torch.bfloat16: dequantize to bfloat16 during compute
# - bnb_4bit_use_double_quant=True: quantize the quantization constants too
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# Load model with quantization (local copy downloaded via HF mirror)
model_name = r"D:\HuggingfaceDownload\DeepSeek-R1-Distill-Qwen-1.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
# Set pad_token to eos_token — causal LMs often don't have a dedicated pad token
tokenizer.pad_token = tokenizer.eos_token

if use_cuda:
    # GPU path: load with 4-bit quantization for maximum memory savings
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
else:
    # CPU fallback: no 4-bit quantization (BitsAndBytes is CUDA-only)
    # Load in bfloat16 instead to save some memory vs float32
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )

# ---------------------------------------------------------------------------
# LoRA (Low-Rank Adaptation) configuration
# ---------------------------------------------------------------------------
# - r=8: rank of the low-rank decomposition matrices (higher = more capacity)
# - lora_alpha=32: scaling factor for the LoRA update (alpha/r = effective lr scale)
# - target_modules=["q_proj", "v_proj"]: apply LoRA to query & value projections
#   in the attention layers — these are the most impactful for adaptation
# - lora_dropout=0.05: dropout on LoRA layers for regularization
# - bias="none": don't train bias terms
# - task_type="CAUSAL_LM": we're fine-tuning a causal (autoregressive) language model
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],   # typical for DeepSeek/Qwen architectures
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Wrap the quantized base model with LoRA adapters
# After this call, only the LoRA parameters will be trained — base weights frozen
model = get_peft_model(model, lora_config)

# Enable gradient checkpointing — trades ~20% slower training for ~30% less VRAM
# by recomputing activations during backward pass instead of storing them
if use_cuda:
    model.gradient_checkpointing_enable()

# Print trainable vs total parameters — expect ~0.1% trainable
model.print_trainable_parameters()  # should show ~0.1% trainable

W0803 19:41:43.117000 12476 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


CUDA available: True


Loading weights: 100%|██████████| 339/339 [00:01<00:00, 273.31it/s]


trainable params: 1,089,536 || all params: 1,778,177,536 || trainable%: 0.0613


## 9. Prepare the training data — with loss masking

We now **tokenize** the formatted text strings into numerical token IDs. We tokenize `train_dataset` and `val_dataset` separately; `test_df` is kept as raw text for unbiased evaluation in Sections 14–15.

The key steps are:

1. **Tokenization**: Each text string is split into tokens using the model's vocabulary, then truncated or padded to exactly 512 tokens. The tokenizer automatically appends `<|endoftext|>` (EOS).

2. **Label creation with loss masking** (THE CRITICAL FIX):
   - All tokens **before** `\nOutput:` are set to `-100` (Hugging Face ignores `-100` in the cross-entropy loss)
   - Only the sentiment label tokens contribute to the loss
   - The model's distribution is now ONLY shaped by the labels, not the template boilerplate

   ```
   "Instruction: ... Output: positive<EOS>[PAD]..."
   labels:   [-100] ...    [-100]  [positive] [<EOS>]  [-100] ...
              └─ ignored ──────┘  └── trained ──┘  └─ ignored ─┘
   ```

3. **Format conversion**: The dataset is set to PyTorch format so tensors are created automatically when batching.

For the full mathematical discussion of how gradients flow through masked tokens via the chain rule, see **Section 9.1** below.

In [4]:
# =============================================================================
# SECTION 9: Tokenize train + val datasets with loss masking
# =============================================================================
# We tokenize train_dataset and val_dataset separately.
# test_df is NOT tokenized — it stays as raw text for inference in Sections 14-15.
#
# KEY FIX — Loss Masking:
#
# PROBLEM (old code: labels = input_ids.clone()):
#   Every token contributed to loss: "Instruction:", "Analyze", "the", etc.
#   The model's ENTIRE statistical distribution learned the template pattern.
#   At inference, max_new_tokens=10 forces 10 tokens of generation. After
#   correctly saying "worry", the remaining budget gets filled with whatever
#   text is statistically most likely from training — which is more template
#   boilerplate (e.g. "\n\nInput: I'm going to miss my").
#
# FIX (loss masking = set pre-Output tokens to -100):
#   The model's distribution is ONLY shaped by the sentiment label portion.
#   Extra generation budget just produces <EOS>/whitespace, not template text.
#
# For the full mathematical discussion of how gradients flow through masked
# tokens via the chain rule, see Section 9.1.

# Encode the output-marker string to find its token IDs
# We'll look for "\nOutput:" in each tokenized sequence to know where to mask
output_marker = "\nOutput:"
output_marker_ids = tokenizer.encode(output_marker, add_special_tokens=False)

def tokenize_function(examples):
    """Convert raw text strings into token IDs with fixed-length padding."""
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors=None
    )

def tokenize_and_label(dataset, desc):
    """Tokenize a raw-text dataset and apply loss masking to labels."""
    tok = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
    tok.set_format("torch", columns=["input_ids", "attention_mask"])

    def set_labels(example):
        """MASK everything before '\nOutput:' with -100. Only the sentiment
        label tokens contribute to the loss. See Section 9.1 for the full math."""
        input_ids = example["input_ids"]
        marker_len = len(output_marker_ids)
        output_start = -1
        for i in range(len(input_ids) - marker_len + 1):
            if input_ids[i:i + marker_len].tolist() == output_marker_ids:
                output_start = i
                break
        if output_start != -1:
            labels = ([-100] * (output_start + marker_len)
                      + input_ids[output_start + marker_len:].tolist())
            labels = labels[:512]
        else:
            labels = [-100] * len(input_ids)
        example["labels"] = labels
        return example

    tok = tok.map(set_labels)
    print(f"{desc}: {len(tok)} tokenized examples")
    return tok

tokenized_train_dataset = tokenize_and_label(train_dataset, "Tokenized Train")
tokenized_val_dataset   = tokenize_and_label(val_dataset,   "Tokenized Val")

Map: 100%|██████████| 5000/5000 [00:02<00:00, 2084.74 examples/s]


Tokenized Train: 5000 tokenized examples


Map: 100%|██████████| 1000/1000 [00:00<00:00, 1976.25 examples/s]

Tokenized Val: 1000 tokenized examples


### 9.1 Understanding Loss Masking — Complete Discussion

This section explains **why** loss masking works, **how** it interacts with backpropagation, and **why it doesn't break alignment**. These are the most frequently misunderstood concepts in instruction fine-tuning.

---

#### What is Loss Masking?

**Loss masking** means setting certain token labels to $-100$ in Hugging Face. The cross-entropy loss function **ignores** positions where `labels == -100` — those positions contribute $0$ to the loss and receive $0$ gradient.

In our code:

```python
labels = [-100] * (output_start + marker_len)  # mask Instruction, Input, Output:
labels.extend(input_ids[output_start + marker_len:])  # keep sentiment label + EOS
```

The masked tokens still flow through the forward pass (self-attention sees them), but the loss is only computed on the sentiment label.

---

#### Was the Model Hallucinating?

**Not in the classical sense** (making up false facts). The model was repeating **statistically learned template patterns**. Because the old code computed loss on *every* token, the model's probability distribution was shaped by the full template: `Instruction:`, `Analyze`, `the`, `Input:`, etc.

At inference, `max_new_tokens=10` forced 10 tokens of generation. After correctly predicting `worry`, the remaining budget was filled with the next-most-likely text the model was trained on — more template boilerplate like `\n\nInput: I'm going to miss my`.

The model wasn't "making things up." It was doing exactly what it was trained to do: predict the most probable continuation given the context.

---

#### Forward Pass vs. Backward Pass — The Key Distinction

This is the single most important concept to understand:

```
┌─── FORWARD PASS (what the model SEES) ───────────────────────────┐
│ Instruction: Analyze ... Input: tweet Output: positive <EOS>     │
│                                                                   │
│ ✓ All tokens flow through self-attention                         │
│ ✓ The hidden state of "positive" is computed from ALL previous   │
│   tokens, including the instruction and the tweet                │
│ ✓ The model builds full context: "after Output:, a label follows"│
└───────────────────────────────────────────────────────────────────┘

┌─── BACKWARD PASS (what the model LEARNS) ────────────────────────┐
│ Instruction: Analyze ... Input: tweet Output: positive <EOS>     │
│    -100      -100          -100   -100    [train]    [train]     │
│                                                                   │
│ ✗ No gradient from Instruction, Input, or Output: markers        │
│ ✓ Gradient ONLY from "positive" and <EOS>                        │
│ ✓ BUT: gradients flow THROUGH attention to update W_K, W_V, W_Q │
│   weights that processed the tweet (via chain rule — see below)  │
└───────────────────────────────────────────────────────────────────┘
```

**The model always reads the full instruction and tweet.** Masking only controls what the model is *graded on*.

---

#### The Chain Rule: Why Masked Tokens Still Get Updated

This is where most people get confused. They think: "If the tweet has `label = -100`, the model ignores the tweet entirely." That is **mathematically false**.

Let's walk through the calculus. Assume:

- Position $j$ = a tweet token (e.g., "love")
- Position $T$ = the sentiment label token (e.g., "positive")
- Only position $T$ contributes to the loss $L$
- Vocabulary size = $V$, hidden dimension = $d$, attention heads = $h$, $d_k = d/h$

**Step 1: The Cross-Entropy Loss with Masking**

For each position $i$, the model produces logits $\mathbf{z}_i \in \mathbb{R}^V$ from the final hidden state $\mathbf{h}_i$ via the output projection $W_{\text{out}} \in \mathbb{R}^{V \times d}$:

$$\mathbf{z}_i = W_{\text{out}} \cdot \mathbf{h}_i + \mathbf{b}_{\text{out}}$$

The loss at position $i$ is the negative log-likelihood:

$$\ell_i = -\log\left(\frac{\exp(z_{i, y_i})}{\sum_{k=1}^{V} \exp(z_{i, k})}\right)$$

where $y_i$ is the true token ID. The total loss is the mean over **non-masked** positions:

$$L = \frac{1}{|\mathcal{M}|} \sum_{i \in \mathcal{M}} \ell_i, \quad \mathcal{M} = \{i : \text{labels}[i] \neq -100\}$$

In our case, $\mathcal{M}$ contains only the positions of the sentiment label and `<EOS>`. All template tokens $(i \notin \mathcal{M})$ contribute $0$ to $L$. The gradient at masked positions is exactly zero:

$$\frac{\partial L}{\partial \mathbf{z}_i} = \mathbf{0} \quad \text{for } i \notin \mathcal{M}$$

But this only means $\frac{\partial L}{\partial \mathbf{z}_i} = 0$ — it does **not** mean $\frac{\partial L}{\partial \mathbf{h}_i} = 0$ (see Step 3).

**Step 2: The attention output at position $T$ is a weighted sum of ALL previous Value vectors**

For each attention head, the output at position $T$ is:

$$O_T = \sum_{i=1}^{T} \alpha_{T,i} V_i$$

Where the attention weights come from softmax of scaled dot products:

$$\alpha_{T,i} = \frac{\exp(s_{T,i})}{\sum_{k=1}^{T} \exp(s_{T,k})}, \quad s_{T,i} = \frac{Q_T \cdot K_i^T}{\sqrt{d_k}}$$

$Q_T = W_Q \cdot \mathbf{h}_T^{(0)}$ is the query at position $T$, $K_i = W_K \cdot \mathbf{h}_i^{(0)}$ is the key at position $i$, and $V_i = W_V \cdot \mathbf{h}_i^{(0)}$ is the value.

Because the tweet at position $j < T$ is one of those indices, $O_T$ **directly depends on** $V_j$:

$$\frac{\partial O_T}{\partial V_j} = \alpha_{T,j} \cdot I_{d_k} \quad \neq 0$$

(The identity matrix appears because $O_T = \sum_i \alpha_{T,i} V_i$ is linear in each $V_i$.)

**Step 3: Gradient flows from the loss to the tweet's Value vector**

Using the chain rule through the output projection and attention:

$$\frac{\partial L}{\partial V_j} = \frac{\partial L}{\partial O_T} \cdot \frac{\partial O_T}{\partial V_j} = \delta_T \cdot \alpha_{T,j}$$

where $\delta_T = \frac{\partial L}{\partial O_T}$ flows from the cross-entropy gradient at position $T$. Since $\delta_T \neq 0$ and $\alpha_{T,j} \neq 0$:

$$\boxed{\frac{\partial L}{\partial V_j} \neq 0}$$

**This is the key insight**: even though position $j$ itself produces no direct loss ($\frac{\partial L}{\partial \mathbf{z}_j} = 0$), the Value vector $V_j$ that *represents* the tweet token receives non-zero gradient because it was *attended to* by the label position.

**Step 4: Gradient flows to the Value weight matrix $W_V$**

Since $V_j = W_V \cdot \mathbf{h}_j^{(0)}$, by the chain rule:

$$\frac{\partial L}{\partial W_V} = \sum_{i=1}^{T} \frac{\partial L}{\partial V_i} \cdot \frac{\partial V_i}{\partial W_V} = \sum_{i=1}^{T} \frac{\partial L}{\partial V_i} \cdot (\mathbf{h}_i^{(0)})^T$$

The tweet token at position $j$ contributes $\frac{\partial L}{\partial V_j} \cdot (\mathbf{h}_j^{(0)})^T$ — a **non-zero** term in the sum.

**Step 5: Full chain to $W_K$ and $W_Q$ (the softmax derivative)**

The attention weight $\alpha_{T,j}$ depends on $K_j$ and all other scores. The softmax derivative is:

$$\frac{\partial \alpha_{T,j}}{\partial s_{T,m}} = \alpha_{T,j}(\delta_{jm} - \alpha_{T,m})$$

where $\delta_{jm}$ is the Kronecker delta. Then:

$$\frac{\partial L}{\partial W_K} = \sum_{i=1}^{T} \frac{\partial L}{\partial \alpha_{T,i}} \cdot \frac{\partial \alpha_{T,i}}{\partial K_i} \cdot \frac{\partial K_i}{\partial W_K}$$

Since $\frac{\partial L}{\partial \alpha_{T,j}} \neq 0$ (via $\frac{\partial L}{\partial O_T}$ and the fact that $O_T$ depends on $\alpha_{T,j}$ through the weighted sum and the softmax normalization), $\frac{\partial L}{\partial W_K} \neq 0$. Same for $W_Q$.

**Step 6: LoRA-specific gradient flow**

In LoRA, the weight update $\Delta W$ is decomposed as:
$$W = W_0 + \Delta W = W_0 + B A$$
where $W_0 \in \mathbb{R}^{d \times d}$ is frozen, $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times d}$, and $r=8$ is the rank.

The forward pass for a LoRA-adapted linear layer is:
$$\mathbf{y} = W_0 \mathbf{x} + \frac{\alpha}{r} B A \mathbf{x}$$

where $\frac{\alpha}{r} = \frac{32}{8} = 4$ is the scaling factor.

Only $A$ and $B$ receive gradients:
$$\frac{\partial L}{\partial B} = \frac{\alpha}{r} \cdot \frac{\partial L}{\partial \mathbf{y}} \cdot (A \mathbf{x})^T$$
$$\frac{\partial L}{\partial A} = \frac{\alpha}{r} \cdot B^T \cdot \frac{\partial L}{\partial \mathbf{y}} \cdot \mathbf{x}^T$$

The gradient $\frac{\partial L}{\partial \mathbf{y}}$ comes from the downstream layers, and as shown above, it flows through attention from the label position back to the tweet positions — updating the LoRA matrices that process the tweet.

**Numerical Example (concrete walkthrough):**

```
Input: "Instruction: Analyze... Input: I love this! Output: positive<EOS>[PAD]..."
Tokens: [I0, I1, ..., Ik, t1, t2, t3, t4, O1, pos, EOS, PAD, PAD, ...]
        └── template + tweet ──┘       └─ label ─┘

loss_contrib: [0, 0, ..., 0, 0, 0, 0, 0,  1,  1, 0, 0, ...]
                                      └─ masked ─┘└ graded ┘

grad_W_K receives contribution from token "pos" WHICH ALSO receives
contribution via attention α_T,j from tokens t1, t2, t3, t4 (the tweet).
```

---

#### Summary Table

| Part of Sequence | In `input_ids`? | In `labels`? | Contributes to Loss? | Receives Gradient? |
|---|---|---|---|---|
| `Instruction:` ... | ✓ Yes | `-100` | ✗ No | ✓ Yes (via attention chain rule) |
| `Input:` (tweet) | ✓ Yes | `-100` | ✗ No | ✓ **Yes** (via attention chain rule) |
| `\nOutput:` marker | ✓ Yes | `-100` | ✗ No | ✓ Yes (via attention chain rule) |
| Sentiment label | ✓ Yes | Real ID | ✓ Yes | ✓ Yes |
| `<EOS>` | ✓ Yes | Real ID | ✓ Yes | ✓ Yes |
| `[PAD]` tokens | ✓ Yes | `-100` | ✗ No | ✗ No (attention mask blocks them) |

**The critical row**: the tweet tokens do not generate their own loss, but the weight matrices that **read** them are updated through backpropagation. The model learns to *interpret* the tweet, not to *reproduce* it.

---

#### Why This is Standard Practice

Loss masking is the **industry norm** for instruction fine-tuning. Here's why:

| Approach | Pros | Cons |
|---|---|---|
| **No masking** (old code) | Simple to implement | Model learns to copy template; hallucinates boilerplate at inference; wastes capacity on unhelpful predictions |
| **Loss masking** (our fix) | Model focuses on the answer; cleaner outputs; standard for LLaMA-2-Chat, Mistral-Instruct, etc. | Slightly more code; requires finding the `\nOutput:` boundary |

---

#### Why Alignment is NOT Broken

Alignment means: **training prompt format ≡ inference prompt format**. Loss masking does not change the prompt format — the model still sees:

```
Instruction: Analyze the sentiment ...\nInput: @user tweet\nOutput:
```

at **both** training and inference time. The model's self-attention mechanism builds the same contextual representations in both cases.

What changes is what the model is *rewarded* for predicting. Without masking, it's rewarded for predicting `Input:` after the label. With masking, it's only rewarded for predicting the label. The *understanding* of the task is the same — only the *output behavior* is different.

**Analogy**: A student reads the full exam question (forward pass). The teacher only grades the final answer (loss calculation). The student still learns to read the question carefully because getting the right answer *depends* on understanding it — the grade on the answer flows back to the study habits (backpropagation through attention).

## 10. Use the standard Hugging Face training workflow

We set up the **data collator** — a utility that dynamically batches tokenized sequences together. `DataCollatorForLanguageModeling` handles:

- **Dynamic padding**: Instead of padding all 5,000 sequences to 512 tokens (wasteful), it pads only within each batch to the longest sequence in that batch.
- **Label preparation**: For causal LM (`mlm=False`), it ensures labels are properly aligned for next-token prediction.
- **Tensor conversion**: Converts lists of token IDs into PyTorch tensors ready for the model.

The cell below imports all remaining Hugging Face classes and creates the data collator.

In [5]:
# =============================================================================
# SECTION 10: Set up the Hugging Face Trainer and data collator
# =============================================================================
# DataCollatorForLanguageModeling dynamically batches tokenized sequences,
# applies padding within each batch, and handles the label shifting for
# causal language modeling (next-token prediction).

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model

# DataCollatorForLanguageModeling: pads sequences to the longest in the batch
# and prepares labels for causal LM training
# mlm=False: we are NOT doing masked language modeling (like BERT)
# — this is causal (autoregressive) LM
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # causal LM — predict next token, not masked token
)

## 11. Set the training arguments (industrial standard)

`TrainingArguments` is the central configuration object that controls every aspect of the training loop. These settings follow the industry standard for LoRA fine-tuning:

| Setting | Value | Explanation |
|---|---|---|
| `seed` | `42` | Deterministic initialization and data shuffling for reproducibility |
| `output_dir` | `./results` | Directory for checkpoints and logs |
| `per_device_train_batch_size` | `4` | 4 samples per GPU per forward pass |
| `gradient_accumulation_steps` | `4` | Accumulate gradients over 4 steps → effective batch = 16 |
| `learning_rate` | `2e-4` | Learning rate for LoRA (higher than full fine-tuning) |
| `weight_decay` | `0.01` | Decoupled weight decay (the "W" in AdamW) |
| `warmup_ratio` | `0.1` | 10% of total steps for LR warmup — prevents early instability |
| `lr_scheduler_type` | `cosine` | Cosine decay schedule (standard for LLM fine-tuning) |
| `max_grad_norm` | `1.0` | Gradient clipping — prevents exploding gradients |
| `fp16` | `True` (GPU) | 16-bit mixed precision — faster, less memory |
| `num_train_epochs` | `3` | Three full passes through the 5,000-sample train set |
| `save_strategy` | `"epoch"` | Save a checkpoint after every epoch |
| `eval_strategy` | `"epoch"` | Evaluate on val set after every epoch |
| `load_best_model_at_end` | `True` | Load the checkpoint with lowest `eval_loss` |
| `metric_for_best_model` | `"eval_loss"` | Use validation loss to pick the best model |
| `optim` | `paged_adamw_8bit` | Memory-efficient 8-bit AdamW (GPU) |
| `report_to` | `"none"` | No external logging (no wandb/tensorboard needed) |

The cell below creates the `TrainingArguments` object with these settings.

In [6]:
# =============================================================================
# SECTION 11: Configure training hyperparameters (industrial standard)
# =============================================================================
# TrainingArguments controls every aspect of the training loop.
# These settings follow the industry standard for LoRA fine-tuning:
# seed, warmup, weight decay, cosine schedule, gradient clipping, eval loop.

import torch
from transformers import TrainingArguments

use_cuda = torch.cuda.is_available()

training_args = TrainingArguments(
    # ---- Reproducibility ----
    seed=42,                              # deterministic weight init & data shuffling

    # ---- Output & logging ----
    output_dir="./results",               # checkpoints and logs saved here
    logging_steps=10,                     # log training loss every 10 steps
    report_to="none",                     # no external logging (wandb/tensorboard)

    # ---- Batch size ----
    per_device_train_batch_size=4,        # 4 samples per GPU per forward pass
    gradient_accumulation_steps=4,        # accumulate gradients over 4 steps
                                          # → effective batch size = 4 × 4 = 16

    # ---- Optimization ----
    learning_rate=2e-4,                   # learning rate for AdamW optimizer
    optim="paged_adamw_8bit" if use_cuda else "adamw_torch",
    weight_decay=0.01,                    # decoupled weight decay (the "W" in AdamW)
    warmup_steps=100,                     # ~10% of total steps (~938) for LR warmup
    lr_scheduler_type="cosine",           # cosine decay (standard for LLM fine-tuning)
    max_grad_norm=1.0,                    # gradient clipping (prevents exploding gradients)

    # ---- Precision ----
    fp16=use_cuda,                        # fp16 mixed precision on GPU (faster, less memory)
    bf16=False,

    # ---- Training duration & evaluation ----
    num_train_epochs=3,                   # 3 full passes through the 5,000-sample train set
    save_strategy="epoch",                # save a checkpoint after each epoch
    eval_strategy="epoch",                # evaluate on val set after each epoch
    load_best_model_at_end=True,          # load the checkpoint with lowest eval_loss
    metric_for_best_model="eval_loss",    # use validation loss to pick best model
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## 12. Start training

The Trainer runs for 3 epochs, logging `train_loss` and `eval_loss` at each epoch. At the end, `load_best_model_at_end=True` automatically restores the checkpoint with the lowest validation loss — protecting against overfitting.

On a 4090-class GPU this takes ~16 minutes (an RTX 5090 laptop GPU is similar). Training on CPU alone is possible but extremely slow.

In [7]:
# =============================================================================
# SECTION 12: Create the Trainer and start fine-tuning
# =============================================================================
# The Hugging Face Trainer handles the entire training loop:
# forward pass → loss → backward pass → optimizer step → logging → eval.
# On a 4090-class GPU this takes ~16 minutes for 3 epochs on 5,000 samples.

# Create the Trainer with model, config, train/val datasets, and data collator
trainer = Trainer(
    model=model,                          # LoRA-wrapped, 4-bit quantized model
    args=training_args,                   # industrial-standard hyperparameters
    train_dataset=tokenized_train_dataset,# tokenized + loss-masked train set
    eval_dataset=tokenized_val_dataset,   # tokenized + loss-masked val set
    data_collator=data_collator,          # dynamic batching & label prep
)

# Start training — Trainer logs train_loss and eval_loss each epoch
# and saves the best checkpoint (lowest eval_loss) automatically
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.891960,2.861261
2,2.784464,2.830182
3,2.748913,2.823386


TrainOutput(global_step=939, training_loss=2.953705788674319, metrics={'train_runtime': 2081.0509, 'train_samples_per_second': 7.208, 'train_steps_per_second': 0.451, 'total_flos': 7.11845609472e+16, 'train_loss': 2.953705788674319, 'epoch': 3.0})

## 13. Save the fine-tuned model

After training completes, we save the **LoRA adapter weights** (NOT the full 1.5B model). The adapter is tiny — typically a few megabytes — because it only contains the low-rank matrices (`A` and `B`) that were added to the attention layers.

The saved directory `./models/DeepSeek1.5B_finetuned/` will contain:
- `adapter_config.json` — LoRA configuration (rank, alpha, target modules, etc.)
- `adapter_model.safetensors` — the trained LoRA weights

At inference time, you load the base model and then attach the adapter with `PeftModel.from_pretrained()`. This is much more disk-efficient than saving a full 3 GB model copy.

In [8]:
# =============================================================================
# SECTION 13: Save the fine-tuned LoRA adapter
# =============================================================================
# Only the LoRA adapter weights are saved (not the full 1.5B base model).
# The adapter is tiny (~a few MB) and can be loaded on top of the base model
# at inference time using PeftModel.from_pretrained().

# Output directory for the LoRA adapter weights
output_dir = "./models/DeepSeek1.5B_finetuned"
# Save the LoRA adapter weights (adapter_config.json + adapter_model.safetensors)
model.save_pretrained(output_dir)
# Save the tokenizer alongside (in case we added special tokens)
tokenizer.save_pretrained(output_dir)
print(f"LoRA adapter saved to {output_dir}")

LoRA adapter saved to ./models/DeepSeek1.5B_finetuned


## 14. Run a demonstration with the fine-tuned model

Now we test the fine-tuned model on a **random tweet from the held-out test set** (`test_df`, never seen during training). The cell below:

1. **Loads the base model** with the same 4-bit quantization used during training
2. **Attaches the LoRA adapter** via `PeftModel.from_pretrained()`
3. **Randomly picks a tweet** from `test_df` — guaranteed unseen during training
4. **Formats the prompt** in the exact same template as training (Section 7)
5. **Generates a prediction** with `temperature=0.1` and `do_sample=True`
6. **Compares predicted vs actual** — prints both labels and a match indicator

> **Note on `do_sample=True`**: For a pure classification task, greedy decoding (`do_sample=False`) is technically optimal. We keep sampling with low temperature (`temperature=0.1`) to demonstrate the generation API — the low temperature makes the output nearly deterministic in practice.

In [9]:
# =============================================================================
# SECTION 14: Run a sentiment analysis demo with the fine-tuned model
# =============================================================================
# This cell demonstrates the end-to-end inference pipeline:
# 1. Load the base model (with same quantization as training)
# 2. Attach the LoRA adapter (the fine-tuned weights)
# 3. Format a tweet in the same instruction template
# 4. Generate a sentiment prediction

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel                        # ← loads LoRA adapter onto base model

# ---- Paths ----
# Base model: the original DeepSeek 1.5B weights (downloaded via HF mirror)
base_model_name = r"D:\HuggingfaceDownload\DeepSeek-R1-Distill-Qwen-1.5B"
# Adapter: the LoRA weights we just fine-tuned and saved
adapter_path = "./models/DeepSeek1.5B_finetuned"

use_cuda = torch.cuda.is_available()

# ---- 4-bit quantization (must match the training configuration) ----
# This is the same BitsAndBytesConfig used during training in Section 8
# Optional: 4-bit quantization (same as during training) — GPU only
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# ---- Load tokenizer from the base model ----
tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
# Set pad_token to eos_token — required for batched generation
tokenizer.pad_token = tokenizer.eos_token   # important for generation

# ---- Load base model (with quantization on GPU, without on CPU) ----
if use_cuda:
    # GPU: load with 4-bit quantization (matches training setup)
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=bnb_config,    # remove if you didn't use quantization
        device_map="auto",
        trust_remote_code=True
    )
else:
    # CPU: cannot use BitsAndBytes; fall back to bfloat16
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )

# ---- Attach the LoRA adapter to the base model ----
# PeftModel.from_pretrained loads the adapter weights and applies them on top
# of the frozen base model. The result behaves like a fine-tuned model.
model = PeftModel.from_pretrained(base_model, adapter_path)

# Switch to evaluation mode — disables dropout, etc.
model.eval()


Loading weights: 100%|██████████| 339/339 [00:01<00:00, 263.17it/s]


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear

In [15]:

# ---- SECTION 14b: Run inference on a RANDOM test-set tweet ----
# test_df is the 1,000 held-out samples from Section 6 — NEVER seen during training.
sample_row = test_df.sample(n=1, random_state=None).iloc[0]  # random selection
tweet = sample_row['content']
actual_sentiment = sample_row['sentiment']

instruction = "Analyze the sentiment of the following tweet:"

# Format prompt EXACTLY as during training (Section 7), ending at "Output:"
prompt = f"Instruction: {instruction}\nInput: {tweet}\nOutput:"

# Tokenize and move to same device as model
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate — eos_token_id tells the model to stop when it produces <EOS>
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=10,
        temperature=0.1,
        do_sample=True,                    # kept for demo; greedy is optimal for classification
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

# Decode only the NEW tokens (skip the prompt)
generated_ids = outputs[0][inputs.input_ids.shape[1]:]
predicted_sentiment = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print(f"Tweet:               {tweet}")
print(f"Actual sentiment:    {actual_sentiment}")
print(f"Predicted sentiment: {predicted_sentiment}")
print(f"Match: {actual_sentiment.lower() == predicted_sentiment.lower()}")

Tweet:               Ugh worried about my math test
Actual sentiment:    worry
Predicted sentiment: worry

Input: worry
Output: worry
Match: False


## 15. Evaluate on the held-out eval set with per-class metrics

This section performs a **rigorous evaluation** on the 1,000 held-out **eval set** (`eval_df` from Section 6). We compute:

- **Accuracy** — overall correctness
- **Precision / Recall / F1** — per-class metrics (positive, negative, neutral)
- **Per-sample comparison** — predicted vs actual for each tweet

The cell below loads the fine-tuned model and runs inference on every eval sample.

In [12]:
# =============================================================================
# SECTION 15: Evaluate on the held-out eval set with per-class metrics
# =============================================================================
# Uses the 1,000-sample eval_df (never seen during training).
# Computes accuracy + per-class precision/recall/F1 via sklearn.

import torch
import os
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_recall_fscore_support
)

# --------------------------
# 1. Paths and model loading
# --------------------------
base_model_name = r"D:\HuggingfaceDownload\DeepSeek-R1-Distill-Qwen-1.5B"
adapter_path = "./models/DeepSeek1.5B_finetuned"

use_cuda = torch.cuda.is_available()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

if use_cuda:
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )

model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

# --------------------------
# 2. Run inference on ALL eval samples
# --------------------------
# eval_df is the 1,000 held-out samples defined in Section 6
instruction = "Analyze the sentiment of the following tweet:"
actuals = []
predictions = []

print(f"Running inference on {len(eval_df)} eval samples...")
for idx, row in eval_df.iterrows():
    tweet = row['content']
    actual = row['sentiment']

    prompt = f"Instruction: {instruction}\nInput: {tweet}\nOutput:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    predicted = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    actuals.append(actual.lower())
    predictions.append(predicted.lower())

# --------------------------
# 3. Compute metrics
# --------------------------
print("\n" + "=" * 60)
print("EVALUATION RESULTS (1,000 held-out eval samples)")
print("=" * 60)

acc = accuracy_score(actuals, predictions)
print(f"\nAccuracy: {acc:.2%}")

print("\n--- Per-Class Metrics ---")
# Use the actual unique classes from the data (not hardcoded)
unique_classes = sorted(set(actuals + predictions))
print(classification_report(
    actuals, predictions,
    labels=unique_classes,
    digits=3
))

# Show a few examples (correct and incorrect)
print("--- Sample Predictions ---")
correct_count = 0
for i in range(len(actuals)):
    if actuals[i] == predictions[i] and correct_count < 3:
        print(f"  ✓ Tweet: {eval_df.iloc[i]['content'][:60]}...")
        print(f"    Actual={actuals[i]}, Predicted={predictions[i]}")
        correct_count += 1
wrong_count = 0
for i in range(len(actuals)):
    if actuals[i] != predictions[i] and wrong_count < 3:
        print(f"  ✗ Tweet: {eval_df.iloc[i]['content'][:60]}...")
        print(f"    Actual={actuals[i]}, Predicted={predictions[i]}")
        wrong_count += 1

Loading weights: 100%|██████████| 339/339 [00:01<00:00, 266.16it/s]


Running inference on 1000 eval samples...

EVALUATION RESULTS (1,000 held-out eval samples)

Accuracy: 0.00%

--- Per-Class Metrics ---
                                                          precision    recall  f1-score   support

                                                   anger      0.000     0.000     0.000       2.0
                                                 boredom      0.000     0.000     0.000       9.0
                                                   empty      0.000     0.000     0.000      23.0
                                              enthusiasm      0.000     0.000     0.000       9.0
                                                     fun      0.000     0.000     0.000      22.0
                                fun

input: @joshuabrown      0.000     0.000     0.000       0.0
                            fun

input: http://bit.ly/wn      0.000     0.000     0.000       0.0
                                               happiness      0.000     0.000  

c:\Users\ycasi\anaconda3\envs\agentic_ai\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ycasi\anaconda3\envs\agentic_ai\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ycasi\anaconda3\envs\agentic_ai\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", resul

## 16. Finish up

1. The LoRA adapter is saved in `./models/DeepSeek1.5B_finetuned` (relative to this notebook folder).

2. The base model lives in `D:\HuggingfaceDownload\DeepSeek-R1-Distill-Qwen-1.5B` — keep it for reuse; there is no need to re-download it.

3. (Optional) Merge the LoRA adapter into the base model to produce a standalone model:

   ```python
   from peft import PeftModel
   merged = PeftModel.from_pretrained(base_model, adapter_path).merge_and_unload()
   merged.save_pretrained("./models/DeepSeek1.5B_finetuned_merged")
   ```

At this point, the example of fine-tuning a billion-parameter model on a single card is complete.

These fine-tuned models can be deployed efficiently locally, greatly reducing both API latency and cost. The same method can be directly extended to tens-of-billions-scale models within 80 GB, such as 7B LLMs.

### Contact

For jobs or project collaboration: `yucongcai_business@outlook.com`  
For research-related inquiries: `yucongcai_research@outlook.com`

---

## Version log

| Version | Date | Change |
|---|---|---|
| v1.0 | 2026-08-03 | Original record of the LoRA fine-tuning notebook (kept in `assets/` as a reference copy). |